# Insight: Does YOB relate to Budget and to Area?

**Data source:** `Cuisine_rating_clean.csv` (the cleaned dataset — no other `_clean_vN` versions exist, so no version choice was needed).

**Relevant columns:** `YOB`, `Budget`, `Location` (used as "area" — see assumptions), `Area code`.

**Question:** Does a diner's year of birth (age) relate to how much they budget for dining, and to which area they dine in?

This runs alongside separate insight notebooks on smoking vs food enjoyment, age vs food/service enjoyment, and cuisine popularity by location/gender. This notebook is scoped only to YOB/Age vs Budget and YOB/Age vs Area.

## Assumptions

- **Age** is derived as `2026 - YOB` (no other date reference exists in the data), used alongside raw `YOB` since "older/younger" is easier to read than birth year.
- **Budget** = the `Budget` column, an ordinal 1-5 scale, treated numerically for correlation.
- **Area** = the `Location` column (human-readable, 8 distinct locations after cleaning: casing/typo/formatting fixes were already applied in `Cuisine_rating_cleaning.ipynb`).
- `Area code` was assumed going in to be redundant with `Location` (1:1). **This assumption turned out to be false — see the check below** — so `Area code` is not used as a second area dimension; `Location` remains the area variable throughout.
- No other filtering — the full 200-row cleaned dataset is used.

In [1]:
import pandas as pd
from scipy import stats
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

cat_cols = ['Location', 'Area code', 'Gender', 'Marital Status', 'Activity', 'Cuisines', 'Alcohol', 'Smoker', 'Often A S']
df = pd.read_csv('Cuisine_rating_clean.csv')
for c in cat_cols:
    df[c] = df[c].astype('category')

df['Age'] = 2026 - df['YOB']
print('Shape:', df.shape)
df[['User ID', 'YOB', 'Age', 'Budget', 'Location', 'Area code']].head()

Shape: (200, 16)


,User ID,YOB,Age,Budget,Location,Area code
0,1,2006,20,3,"Upper East Side,NY",153
1,2,1991,35,3,"St. George,NY",123
2,3,1977,49,5,"Upper West Side,NY",122
3,4,1956,70,5,"Upper East Side,NY",153
4,5,1997,29,4,"Central Park,NY",129


In [2]:
# Check the assumption that Area code is 1:1 with Location
codes_per_location = df.groupby('Location', observed=True)['Area code'].nunique().sort_values(ascending=False)
print('Distinct Area codes per Location:')
print(codes_per_location)
print('\n1:1 with Location?', (codes_per_location == 1).all())

Distinct Area codes per Location:
Location
St. George,NY         22
Upper East Side,NY    16
Central Park,NY       15
Riverdale,NY          13
China Town,NY         12
Market City,NY        11
Upper West Side,NY    10
Cedar Hill,NY          2
Name: Area code, dtype: int64

1:1 with Location? False


**Correction to the stated assumption:** `Area code` is *not* redundant with `Location` — every location has many distinct area codes (e.g. `St. George,NY` alone spans 22 different codes across its 46 rows). `Area code` looks like a finer-grained code (block/building level) while `Location` is the neighborhood. This doesn't change the analysis plan (`Location` is still used as "area"), but it means `Area code` carries information `Location` doesn't — worth flagging as a follow-up, not investigated further here.

## Part 1 — YOB/Age vs Budget

In [3]:
r_age, p_age = stats.pearsonr(df['Age'], df['Budget'])
rho_age, p_rho = stats.spearmanr(df['Age'], df['Budget'])
print(f'Pearson  r(Age, Budget)     = {r_age:.3f}   (p={p_age:.3f})')
print(f'Spearman rho(Age, Budget)  = {rho_age:.3f}   (p={p_rho:.3f})')

Pearson  r(Age, Budget)     = 0.071   (p=0.315)
Spearman rho(Age, Budget)  = 0.053   (p=0.454)


In [4]:
bins = [17, 24, 34, 44, 54, 64, 72]
labels = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
df['AgeBracket'] = pd.cut(df['Age'], bins=bins, labels=labels, include_lowest=True)

budget_by_bracket = df.groupby('AgeBracket', observed=True)['Budget'].agg(['count', 'mean', 'median', 'std']).round(2)
budget_by_bracket

,count,mean,median,std
AgeBracket,,,,
18-24,40,3.90,4.0,0.84
25-34,44,3.36,3.5,1.24
35-44,28,4.32,4.5,0.77
45-54,36,3.83,4.0,1.13
55-64,28,3.64,3.5,1.13
65+,24,4.08,4.0,0.78


In [5]:
bracket_groups = [g['Budget'].values for _, g in df.groupby('AgeBracket', observed=True)]
f_stat, f_p = stats.f_oneway(*bracket_groups)
print(f'One-way ANOVA, Budget across age brackets: F={f_stat:.3f}, p={f_p:.4f}')

One-way ANOVA, Budget across age brackets: F=3.630, p=0.0037


**Part 1 finding:** More subtle than a first glance suggests. The *linear* Age-Budget correlation is essentially zero (Pearson r=0.07, p=0.32; Spearman rho=0.05, p=0.45), which would normally read as "no relationship." But the one-way ANOVA across the 6 age brackets **is significant** (F=3.63, p=0.0037) — because the relationship isn't monotonic. The 25-34 bracket budgets noticeably lower (mean 3.36) than every other bracket (which cluster in the 3.6-4.3 range, with 35-44 highest at 4.32). A linear correlation can't see this dip-then-recover shape; the bracket comparison can. So: age *does* relate to budget, but through a "25-34 spends less" effect rather than a steady age trend.

## Part 2 — YOB/Age vs Area (Location)

In [6]:
age_by_location = df.groupby('Location', observed=True)['Age'].agg(['count', 'mean', 'median', 'std']).sort_values('mean').round(2)
age_by_location

,count,mean,median,std
Location,,,,
"Central Park,NY",32,35.69,27.5,16.92
"St. George,NY",46,39.57,38.0,15.91
"Upper East Side,NY",30,40.47,39.0,17.33
"Upper West Side,NY",18,41.22,35.0,16.98
"China Town,NY",22,42.18,49.0,16.10
"Riverdale,NY",28,42.29,43.5,17.40
"Market City,NY",22,48.73,52.0,15.57
"Cedar Hill,NY",2,66.00,66.0,0.00


In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

order = age_by_location.index.tolist()
data_by_loc = [df.loc[df['Location'] == loc, 'Age'].values for loc in order]

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot(data_by_loc, tick_labels=order)
ax.set_ylabel('Age')
ax.set_title('Age distribution by Location')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

/var/folders/sz/3_np9r_91db6_dhc9819tkwh0000gn/T/ipykernel_82279/3842096456.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
location_groups = [g['Age'].values for _, g in df.groupby('Location', observed=True)]
f_stat_loc, f_p_loc = stats.f_oneway(*location_groups)
h_stat_loc, h_p_loc = stats.kruskal(*location_groups)
print(f'One-way ANOVA,  Age across locations: F={f_stat_loc:.3f}, p={f_p_loc:.4f}')
print(f'Kruskal-Wallis, Age across locations: H={h_stat_loc:.3f}, p={h_p_loc:.4f}')

One-way ANOVA,  Age across locations: F=1.900, p=0.0715
Kruskal-Wallis, Age across locations: H=12.940, p=0.0736


**Part 2 finding:** A mild, marginal pattern — `Central Park,NY` skews youngest (mean age 35.7) and `Market City,NY` skews oldest (mean age 48.7), a ~13-year gap. But neither the ANOVA (p=0.07) nor the Kruskal-Wallis test (p=0.07) reaches conventional significance (p<0.05); this is a trend worth noting, not a confirmed effect.

## Confidence & caveats

- **Budget (Part 1):** well-powered (n=200 across 6 brackets, smallest bracket n=24) and the bracket effect is statistically significant (p=0.0037), not a coin flip. Caveat: with 6 brackets there's some multiple-comparison risk (more groups tested = higher chance one looks different by chance), and this notebook didn't run a post-hoc test (e.g. Tukey HSD) to confirm *which* specific bracket pairs differ — the headline "25-34 is the low outlier" is visually clear from the means table but not individually significance-tested pair-by-pair.
- **Area (Part 2):** `Cedar Hill,NY` has only **2 rows** (both age 66, hence std=0) — its "oldest tied with Market City" position is not statistically meaningful, just 2 data points. Excluding it, the ANOVA/Kruskal-Wallis p-values (both ≈0.07) would likely shift somewhat; the pattern should be read as suggestive, not confirmed, given it's already borderline with the small location included.
- Both parts use the full 200-row dataset with no missing data (per the cleaning pass), so there's no missing-data bias to worry about.

## Conclusion

**YOB/Age vs Budget: a real but non-linear relationship.** There's no straight-line trend (correlation ≈0, not significant), but budget genuinely differs by age bracket (ANOVA p=0.0037): 25-34-year-olds budget noticeably lower (mean 3.36/5) than every other age group, which otherwise sit in a fairly flat 3.6-4.3 range. So age matters, but specifically through a dip in the 25-34 bracket rather than a rising or falling trend with age.

**YOB/Age vs Area: a mild, unconfirmed trend.** `Central Park,NY` diners skew younger (mean 35.7) and `Market City,NY` diners skew older (mean 48.7) — a real-looking ~13-year gap — but the difference across all 8 locations does not clear the conventional significance bar (p≈0.07 on both ANOVA and Kruskal-Wallis), so this should be treated as a lead worth watching, not an established finding.

## Flagged follow-ups

- **`Area code` is not redundant with `Location`** (contrary to the scoping assumption) — it looks like a finer-grained code within each neighborhood. Worth its own insight if there's a reason to care about sub-location granularity (e.g. does age or budget vary *within* a location by area code?).
- The Location-vs-Age trend is right at the edge of significance (p≈0.07) and is influenced by one 2-row location (`Cedar Hill,NY`). If this pattern matters for a decision, worth flagging for more data collection at that location rather than treating the current read as settled.